Part A: permissions

In [0]:
---Cell 1: who can do what right now?
SHOW GRANTS ON SCHEMA workspace.finance;
SHOW GRANTS ON TABLE workspace.finance.silver_transactions;

Principal,ActionType,ObjectType,ObjectKey


Principal,ActionType,ObjectType,ObjectKey


In [0]:
-- Imagine a "finance_readers" group for family members or an accountant.
-- In Free Edition we use the built-in group `account users` to practise the syntax.
GRANT USE CATALOG ON CATALOG workspace TO `account users`;
GRANT USE SCHEMA ON SCHEMA workspace.finance TO `account users`;
GRANT SELECT ON TABLE workspace.finance.gold_monthly_summary TO `account users`;

SHOW GRANTS ON TABLE workspace.finance.gold_monthly_summary;

Principal,ActionType,ObjectType,ObjectKey
account users,SELECT,TABLE,workspace.finance.gold_monthly_summary


In [0]:
---Cell 3: take it away again
REVOKE SELECT ON TABLE workspace.finance.gold_monthly_summary FROM `account users`;
SHOW GRANTS ON TABLE workspace.finance.gold_monthly_summary;

Principal,ActionType,ObjectType,ObjectKey


Part B: column masks (hide sensitive values)

In [0]:
---Cell 4: a masking function
CREATE OR REPLACE FUNCTION workspace.finance.mask_bank_ref(ref STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('finance_auditors') THEN ref          -- auditors see everything
  ELSE concat('********', right(ref, 4))                             -- everyone else: last 4 digits
END;

In [0]:
---Cell 5: attach it to the column
ALTER TABLE workspace.finance.silver_transactions
  ALTER COLUMN bank_ref SET MASK workspace.finance.mask_bank_ref;

SELECT txn_date, description, bank_ref
FROM workspace.finance.silver_transactions
WHERE bank_ref IS NOT NULL LIMIT 5;

txn_date,description,bank_ref
2026-09-03,UPI/SWIGGY/swiggy@axisbank/Food order/554302504,********8950
2026-09-04,UPI/CULT FIT/cultfit@razorpay/Membership/641819166,********6648
2026-08-07,UPI/CULT FIT/cultfit@razorpay/Membership/724081856,********7200
2026-07-26,UPI/SRI LAKSHMI TEA STALL/paytmqr@paytm/Payment/270541592,********3555
2026-05-05,IMPS-345877882397-RAJESH KUMAR-HOUSE RENT,********5356


Part C: row filters (hide whole rows)

In [0]:
---Cell 6: a row filter
CREATE OR REPLACE FUNCTION workspace.finance.only_card_rows(account STRING)
RETURNS BOOLEAN
RETURN is_account_group_member('finance_auditors') OR account = 'bank_b';

ALTER TABLE workspace.finance.silver_transactions
  SET ROW FILTER workspace.finance.only_card_rows ON (account);

SELECT account, count(*) FROM workspace.finance.silver_transactions GROUP BY account;

account,count(1)
bank_b,88


In [0]:
---Cell 7: remove both, so your later modules work normally
ALTER TABLE workspace.finance.silver_transactions DROP ROW FILTER;
ALTER TABLE workspace.finance.silver_transactions ALTER COLUMN bank_ref DROP MASK;

SELECT account, count(*) FROM workspace.finance.silver_transactions GROUP BY account;   -- both banks again

account,count(1)
bank_a,189
bank_b,88


Part D: comments and tags (documentation)

In [0]:
---Cell 8: describe your tables and columns
COMMENT ON TABLE workspace.finance.silver_transactions IS
  'One row per real transaction from all accounts. De-duplicated across overlapping monthly exports.';

ALTER TABLE workspace.finance.silver_transactions
  ALTER COLUMN txn_id COMMENT 'SHA-256 of account, date, description, amount, direction, ref and occurrence-within-file';

ALTER TABLE workspace.finance.silver_transactions
  ALTER COLUMN amount COMMENT 'Always positive. Use `direction` (DEBIT/CREDIT) for the sign.';

In [0]:
---Cell 9: tag what’s sensitive
ALTER TABLE workspace.finance.silver_transactions
  SET TAGS ('domain' = 'personal_finance', 'layer' = 'silver');

ALTER TABLE workspace.finance.silver_transactions
  ALTER COLUMN description SET TAGS ('pii' = 'merchant_and_payee_names');

ALTER TABLE workspace.finance.silver_transactions
  ALTER COLUMN bank_ref SET TAGS ('pii' = 'bank_reference');

Part E: metadata as data (information_schema)

In [0]:
---Cell 10: every table in your schema, with its comment
SELECT table_name, table_type, comment
FROM workspace.information_schema.tables
WHERE table_schema = 'finance'
ORDER BY table_name;

table_name,table_type,comment
ai_merchant_categories,MANAGED,null
bank_a_clean,MANAGED,null
bank_b_clean,MANAGED,null
bronze_bank_a,MANAGED,null
bronze_bank_b,MANAGED,null
budget_alerts,MANAGED,null
budgets,MANAGED,null
gold_budget_status,VIEW,null
gold_category_spend,VIEW,null
gold_monthly_summary,MANAGED,null


In [0]:
---Cell 11: find every PII column you’ve tagged
SELECT schema_name, table_name, column_name, tag_name, tag_value
FROM workspace.information_schema.column_tags
WHERE tag_name = 'pii';

schema_name,table_name,column_name,tag_name,tag_value
finance,silver_transactions,description,pii,merchant_and_payee_names
finance,silver_transactions,bank_ref,pii,bank_reference


Part F: system tables (Databricks’ own logs)

In [0]:
---Cell 12: lineage as a table
SELECT source_table_full_name, target_table_full_name, entity_type, event_time
FROM system.access.table_lineage
WHERE target_table_full_name LIKE 'workspace.finance%'
ORDER BY event_time DESC
LIMIT 20;

source_table_full_name,target_table_full_name,entity_type,event_time
null,workspace.finance.budgets,DBSQL_QUERY,2026-11-09T04:51:56.505Z
workspace.finance_pipeline.gold_budget_status,workspace.finance.budget_alerts,JOB,2026-11-09T04:49:38.276Z
workspace.finance_pipeline.gold_budget_status,workspace.finance.budget_alerts,JOB,2026-11-09T04:49:35.291Z
null,workspace.finance_pipeline.event_log_6728de8e_9412_41ef_b518_9abe65c9e665,PIPELINE,2026-11-09T04:48:39.500Z
workspace.finance.budgets,workspace.finance_pipeline.gold_budget_status,PIPELINE,2026-11-09T04:48:38.094Z
workspace.finance_pipeline.gold_category_spend,workspace.finance_pipeline.gold_budget_status,PIPELINE,2026-11-09T04:48:38.094Z
workspace.finance_pipeline.transactions_categorized,workspace.finance_pipeline.gold_monthly_summary,PIPELINE,2026-11-09T04:48:35.481Z
workspace.finance_pipeline.transactions_categorized,workspace.finance_pipeline.gold_category_spend,PIPELINE,2026-11-09T04:48:35.437Z
null,workspace.finance_pipeline.event_log_6728de8e_9412_41ef_b518_9abe65c9e665,PIPELINE,2026-11-09T04:48:32.401Z
workspace.finance.ai_merchant_categories,workspace.finance_pipeline.transactions_categorized,PIPELINE,2026-11-09T04:48:32.059Z


In [0]:
---Cell 13: who queried what (audit log)
SELECT event_time, user_identity.email, action_name, request_params
FROM system.access.audit
WHERE service_name = 'unityCatalog' AND event_date >= current_date() - 1
ORDER BY event_time DESC
LIMIT 20;

event_time,email,action_name,request_params
2026-11-09T09:11:31.480Z,mmaverick254@gmail.com,getUserMappedCredential,"Map(name_arg -> system.ai.microsoft_365, workspace_id -> 7474648742261623, metastore_id -> b2ad005e-47fd-4686-8dda-68d08447f1d2)"
2026-11-09T09:11:31.476Z,mmaverick254@gmail.com,getUserMappedCredential,"Map(name_arg -> system.ai.gmail, workspace_id -> 7474648742261623, metastore_id -> b2ad005e-47fd-4686-8dda-68d08447f1d2)"
2026-11-09T09:11:31.474Z,mmaverick254@gmail.com,getUserMappedCredential,"Map(name_arg -> system.ai.atlassian, workspace_id -> 7474648742261623, metastore_id -> b2ad005e-47fd-4686-8dda-68d08447f1d2)"
2026-11-09T09:11:31.472Z,mmaverick254@gmail.com,getUserMappedCredential,"Map(name_arg -> system.ai.google_drive, workspace_id -> 7474648742261623, metastore_id -> b2ad005e-47fd-4686-8dda-68d08447f1d2)"
2026-11-09T09:11:31.472Z,mmaverick254@gmail.com,getUserMappedCredential,"Map(name_arg -> system.ai.github, workspace_id -> 7474648742261623, metastore_id -> b2ad005e-47fd-4686-8dda-68d08447f1d2)"
2026-11-09T09:11:31.471Z,mmaverick254@gmail.com,getUserMappedCredential,"Map(name_arg -> system.ai.google_calendar, workspace_id -> 7474648742261623, metastore_id -> b2ad005e-47fd-4686-8dda-68d08447f1d2)"
2026-11-09T09:11:31.471Z,mmaverick254@gmail.com,getUserMappedCredential,"Map(name_arg -> system.ai.slack, workspace_id -> 7474648742261623, metastore_id -> b2ad005e-47fd-4686-8dda-68d08447f1d2)"
2026-11-09T09:11:30.613Z,mmaverick254@gmail.com,getConnection,"Map(parent -> schemas/system.ai, name_arg -> system.ai.microsoft_365, dependent -> {""mcp_service"":{""id"":""f8ad802a-068b-4dcf-b312-1296f5d6c9ed""}}, workspace_id -> 7474648742261623, metastore_id -> b2ad005e-47fd-4686-8dda-68d08447f1d2)"
2026-11-09T09:11:30.613Z,mmaverick254@gmail.com,getConnection,"Map(parent -> schemas/system.ai, name_arg -> system.ai.slack, dependent -> {""mcp_service"":{""id"":""3856c871-62fa-4999-828b-fae00bebfda3""}}, workspace_id -> 7474648742261623, metastore_id -> b2ad005e-47fd-4686-8dda-68d08447f1d2)"
2026-11-09T09:11:30.613Z,mmaverick254@gmail.com,getConnection,"Map(parent -> schemas/system.ai, name_arg -> system.ai.gmail, dependent -> {""mcp_service"":{""id"":""94724e1f-caca-4928-9ece-306edda9f0f3""}}, workspace_id -> 7474648742261623, metastore_id -> b2ad005e-47fd-4686-8dda-68d08447f1d2)"


In [0]:
---Cell 14: how much compute your work used
SELECT usage_date, billing_origin_product, sum(usage_quantity) AS dbus
FROM system.billing.usage
WHERE usage_date >= current_date() - 30
GROUP BY ALL
ORDER BY usage_date DESC, dbus DESC;

usage_date,billing_origin_product,dbus
2026-11-09,DLT,1.670254256944444562
2026-11-09,SQL,1.230141111111111111
2026-11-09,JOBS,0.568294991071428663
2026-11-09,GENIE,0.204133000000000000
2026-11-08,DLT,1.037405833333333339
2026-11-08,PREDICTIVE_OPTIMIZATION,0.146926955208333350
2026-11-08,DEFAULT_STORAGE,0.072650000000000000
2026-11-07,DLT,1.095887305555555610
2026-11-07,DEFAULT_STORAGE,0.081050959274696977
2026-11-06,DLT,1.089691895833333334
